In [4]:
import pandas as pd

In [6]:
df = pd.read_csv("../data/Recipe_Clean.csv")

text = "".join(df["text"])

In [7]:
characters = sorted(list(set(text)))
vocab_size = len(characters)

print("Vocabulary:")
print("".join(characters))
print("Vocabulary Size:", vocab_size)

Vocabulary:

 !"#%&'()*+,-./0123456789:;?ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]abcdefghijklmnopqrstuvwxyz{}®°ÉÎÚßàáâãäåçèéêëìíîñòóôöùúûüąĆćčĐł̀̂ậị​‑–—‘⁄℉™
Vocabulary Size: 133


In [8]:
print(text[text.find("¼")-50:text.find("¼")+50])

In [9]:
from collections import Counter

counter = Counter(text)

for char, count in counter.items():
    if ord(char) > 127:
        print(repr(char), count)

'®' 5497
'™' 181
'é' 2986
'°' 103
'ñ' 2548
'â' 80
'—' 150
'è' 412
'û' 73
'ã' 8
'–' 48
'ú' 19
'⁄' 87
'á' 33
'ì' 5
'ó' 41
'í' 23
'ê' 486
'Î' 4
'î' 45
'Ć' 11
'č' 11
'ć' 11
'ò' 9
'ị' 1
'à' 19
'Đ' 1
'ậ' 1
'‑' 8
'ö' 56
'É' 17
'ë' 9
'ä' 13
'‘' 9
'ß' 4
'ü' 7
'Ú' 5
'\u200b' 18
'ç' 20
'ô' 1
'å' 4
'℉' 2
'ù' 3
'̂' 2
'̀' 2
'ł' 1
'ą' 1


In [10]:
# tokenize - create a mapping between characters to integers
char_to_idx = { ch:i for i,ch in enumerate(characters) }
idx_to_char = { i:ch for i,ch in enumerate(characters) }
encode = lambda xs: [char_to_idx[x] for x in xs] # encoder: take the string, output the list of integers
decode = lambda xs: ''.join([idx_to_char[x] for x in xs]) # decoder: take the list of integers, output the string

In [11]:
sample = text[:200]
print(sample)

encoded = encode(sample)

print(encoded[:50])

decoded = decode(encoded)

print(decoded)

Title: Air Fryer Potato Slices with Dipping Sauce

Ingredients: 3/4 cup ketchup
1/2 cup beer
1 tablespoon Worcestershire sauce
1/2 teaspoon onion powder
1/4 teaspoon cayenne
2 baking potatoes
olive oi
[48, 66, 77, 69, 62, 26, 1, 29, 66, 75, 1, 34, 75, 82, 62, 75, 1, 44, 72, 77, 58, 77, 72, 1, 47, 69, 66, 60, 62, 76, 1, 80, 66, 77, 65, 1, 32, 66, 73, 73, 66, 71, 64, 1, 47, 58, 78, 60, 62, 0]
Title: Air Fryer Potato Slices with Dipping Sauce

Ingredients: 3/4 cup ketchup
1/2 cup beer
1 tablespoon Worcestershire sauce
1/2 teaspoon onion powder
1/4 teaspoon cayenne
2 baking potatoes
olive oi


In [12]:
%pip install tiktoken

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [13]:
import tiktoken

enc = tiktoken.get_encoding("gpt2")

tokens = enc.encode(text)

print(tokens[:20])
print(enc.decode(tokens[:20]))

[19160, 25, 3701, 25305, 263, 43876, 311, 677, 274, 351, 360, 4501, 37618, 198, 198, 41222, 25, 513, 14, 19]
Title: Air Fryer Potato Slices with Dipping Sauce

Ingredients: 3/4


In [14]:
print("Character vocab size:", len(characters))
print("GPT-2 vocab size:", enc.n_vocab)

Character vocab size: 133
GPT-2 vocab size: 50257


In [15]:
sample1 = text[:200]

tokens = enc.encode(sample)

print("Number of characters:", len(sample))
print("Number of tokens:", len(tokens))

print(enc.decode(tokens))

Number of characters: 200
Number of tokens: 61
Title: Air Fryer Potato Slices with Dipping Sauce

Ingredients: 3/4 cup ketchup
1/2 cup beer
1 tablespoon Worcestershire sauce
1/2 teaspoon onion powder
1/4 teaspoon cayenne
2 baking potatoes
olive oi


Byte-Pair Encoding

In [16]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [17]:
def merge(ids, pair, idx):
  newids = []
  i = 0
  while i < len(ids):
    if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
      newids.append(idx)
      i += 2
    else:
      newids.append(ids[i])
      i += 1
  return newids

In [18]:
bpe_vocab_size = 300 # the desired final vocabulary size (must be larger than the base char vocab)
num_merges = bpe_vocab_size - vocab_size # merges only ADD tokens, so this grows vocab_size up to bpe_vocab_size
tokens = encode(text)
ids = list(tokens) # copy so we don't destroy the original list

merges = {} # (int, int) -> int
for i in range(num_merges):
  stats = get_stats(ids)
  pair = max(stats, key=stats.get)
  idx = vocab_size + i
  print(f"merging {pair} into a new token {idx}")
  ids = merge(ids, pair, idx)
  merges[pair] = idx

merging (62, 1) into a new token 133
merging (66, 71) into a new token 134
merging (1, 77) into a new token 135
merging (61, 1) into a new token 136
merging (1, 58) into a new token 137
merging (62, 75) into a new token 138
merging (72, 71) into a new token 139
merging (75, 62) into a new token 140
merging (1, 60) into a new token 141
merging (62, 76) into a new token 142
merging (1, 76) into a new token 143
merging (62, 58) into a new token 144
merging (71, 136) into a new token 145
merging (66, 69) into a new token 146
merging (72, 78) into a new token 147
merging (62, 71) into a new token 148
merging (1, 59) into a new token 149
merging (1, 70) into a new token 150
merging (14, 1) into a new token 151
merging (58, 75) into a new token 152
merging (135, 65) into a new token 153
merging (135, 72) into a new token 154
merging (73, 72) into a new token 155
merging (137, 145) into a new token 156
merging (0, 17) into a new token 157
merging (77, 1) into a new token 158
merging (134, 64) 

KeyboardInterrupt: 

In [25]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 65097605
ids length: 32787973
compression ratio: 1.99X


In [54]:
print("tokens length:", len(tokens))
print("ids length:", len(ids))
print(f"compression ratio: {len(tokens) / len(ids):.2f}X")

tokens length: 65103283
ids length: 42170441
compression ratio: 1.54X


In [27]:
vocab = idx_to_char.copy()
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

In [28]:
for i in range(vocab_size, vocab_size + 30):
    print(i, repr(vocab[i]))

134 'e '
135 'in'
136 ' t'
137 'd '
138 ' a'
139 'er'
140 'on'
141 're'
142 ' c'
143 'es'
144 ' s'
145 'ea'
146 'nd '
147 'il'
148 'ou'
149 'en'
150 ' b'
151 ' m'
152 '\n\n'
153 '. '
154 'ar'
155 ' th'
156 ' to'
157 'po'
158 ' and '
159 '\n1'
160 't '
161 'ing'
162 ' the '
163 ', '


In [29]:
for i in range(max(vocab.keys()) - 30, max(vocab.keys()) + 1):
    print(i, repr(vocab[i]))

269 'is'
270 'ction'
271 'heat'
272 ' in a'
273 'owl'
274 'Di'
275 '. S'
276 'ick'
277 'ion'
278 '\n1/2'
279 'oun'
280 'oven'
281 ' g'
282 'bout '
283 'id'
284 'e s'
285 ' mix'
286 'au'
287 'der'
288 'igh'
289 'rection'
290 'utter'
291 'Ingre'
292 ' lar'
293 'RE'
294 'EN'
295 'le: '
296 'IP'
297 'Ingredient'
298 'Tit'
299 'END'


In [31]:
def decode_bpe(ids):
  # given ids (list of integers), return Python string
  text = "".join(vocab[idx] for idx in ids)
  return text

In [32]:
def encode_bpe(text):
  # given a string, return list of integers (the tokens)
  tokens = list(encode(text))
  while len(tokens) >= 2:
    stats = get_stats(tokens)
    pair = min(stats, key=lambda p: merges.get(p, float("inf")))
    if pair not in merges:
      break # nothing else can be merged
    idx = merges[pair]
    tokens = merge(tokens, pair, idx)
  return tokens

In [33]:
sample = text[:500]

encoded = encode_bpe(sample)

print("Characters:", len(sample))
print("BPE tokens:", len(encoded))

Characters: 500
BPE tokens: 235
